# Controlli aggiuntivi del modello nominale

Due controlli promessi nella proposta e non presenti in `Bayesian_Nominal.ipynb` (che non viene rieseguito):

1. **Posterior predictive check** della distribuzione di massa: per $\theta$ estratti dalla posterior si genera un dataset replicato e si confronta una discrepanza $T(y,\theta)$ calcolata sui dati e sulle repliche; $p_B=P(T_{\rm rep}\ge T_{\rm obs})$. Discrepanze: $\chi^2$ binnato (bin da 10 MeV, entrambe le cariche), asimmetria di conteggio nell'intera finestra, nella finestra di segnale e nelle sideband.
2. **Trace plot**: catena del modello nominale con walker inizializzati uniformemente nella scatola della prior (25000 passi), salvata ogni 10 passi.

Legge `results/nominal/nominal_posterior_thinned.npy` e `nominal_summary.json`; scrive `results/nominal/ppc.json` e `results/nominal/trace_dispersed.npz`. Ogni blocco riprende se il file esiste.

3. **Dalitz per polarità del magnete**: la componente sinistra-destra dell'asimmetria di rivelazione cambia segno con la polarità; se la struttura locale è la stessa nei due campioni non è dovuta a essa. Scrive `results/nominal/dalitz_magnet.json`.
4. **Risonanze nel Dalitz**: asimmetria di conteggio nella banda della $\phi(1020)$ e nella regione a bassa massa fuori dalla $\phi$. Scrive `results/nominal/dalitz_resonances.json`.


In [1]:
import os, sys, json
import numpy as np
sys.path.insert(0, ".")
import bayesian_nominal as bn
import emcee

DATA_DIR = "../../opendata-project/Data"
RESULTS_DIR = os.path.join(os.getcwd(), "results", "nominal")
cfg = bn.NOMINAL
data = bn.select(bn.load_candidates(DATA_DIR), cfg)
m_obs, q_obs = data.m.to_numpy(), data.q.to_numpy()
summary = json.load(open(os.path.join(RESULTS_DIR, "nominal_summary.json")))
names = summary["nominal"]["names"]
thinned = np.load(os.path.join(RESULTS_DIR, "nominal_posterior_thinned.npy"))
MU, SIG = summary["nominal"]["mu"]["median"], summary["nominal"]["sigma"]["median"]
print(len(m_obs), thinned.shape)


19731 (20000, 12)


## Posterior predictive check

In [2]:
N_REP = 1000
def count_asym(q):
    return float((np.sum(q < 0) - np.sum(q > 0)) / len(q)) if len(q) else float("nan")
def discrepancies(m, q, theta):
    sw = np.abs(m - MU) < 2 * SIG
    sb = np.abs(m - MU) > 4 * SIG
    g = bn.binned_gof(bn.Posterior(m, q, cfg), theta)
    return {"chi2": g["chi2"], "chi2_pos": g["per_charge"]["pos"], "chi2_neg": g["per_charge"]["neg"],
            "A_window": count_asym(q), "A_signal_window": count_asym(q[sw]), "A_sidebands": count_asym(q[sb])}

path = os.path.join(RESULTS_DIR, "ppc.json")
if os.path.exists(path):
    ppc = json.load(open(path)); print("[resume]", path)
else:
    rng = np.random.default_rng(2026)
    idx = rng.choice(len(thinned), N_REP, replace=False)
    rows = []
    for k, i in enumerate(idx):
        th = thinned[i]
        m_rep, q_rep = bn.generate_toy(dict(zip(names, th)), cfg, rng)
        rows.append({"obs": discrepancies(m_obs, q_obs, th), "rep": discrepancies(m_rep, q_rep, th)})
        if (k + 1) % 200 == 0:
            print(k + 1)
    keys = list(rows[0]["obs"])
    ppc = {"n_rep": N_REP, "rows": rows, "p_B": {}}
    for key in keys:
        o = np.array([r["obs"][key] for r in rows]); r_ = np.array([r["rep"][key] for r in rows])
        if key.startswith("chi2"):
            ppc["p_B"][key] = float(np.mean(r_ >= o))           # one-sided: misfit = large chi2
        else:
            ppc["p_B"][key] = float(2 * min(np.mean(r_ >= o), np.mean(r_ <= o)))   # two-sided
    json.dump(ppc, open(path, "w"), indent=1); print("[saved]", path)
for key, p in ppc["p_B"].items():
    o = np.array([r["obs"][key] for r in ppc["rows"]]); r_ = np.array([r["rep"][key] for r in ppc["rows"]])
    print(f"{key:16s} T_obs = {np.median(o):9.4f}   T_rep = {r_.mean():9.4f} +- {r_.std():.4f}   p_B = {p:.3f}")


[resume] /Users/gmoretti/Work/wt_bayes/LHCb/Code_general/Bayesian/results/nominal/ppc.json
chi2             T_obs =  174.6950   T_rep =  103.8398 +- 14.7864   p_B = 0.000
chi2_pos         T_obs =  100.5458   T_rep =   51.8853 +- 10.3877   p_B = 0.000
chi2_neg         T_obs =   73.5755   T_rep =   51.9545 +- 10.4754   p_B = 0.030
A_window         T_obs =   -0.0432   T_rep =   -0.0432 +- 0.0098   p_B = 0.996
A_signal_window  T_obs =   -0.0461   T_rep =   -0.0465 +- 0.0110   p_B = 0.992
A_sidebands      T_obs =   -0.0106   T_rep =   -0.0182 +- 0.0246   p_B = 0.734


## Trace plot (partenza dispersa)

In [3]:
path_tr = os.path.join(RESULTS_DIR, "trace_dispersed.npz")
if os.path.exists(path_tr):
    print("[resume]", path_tr)
else:
    post = bn.Posterior(m_obs, q_obs, cfg)
    lo_, hi_ = bn._prior_box(cfg, post.n, names)
    rng = np.random.default_rng(7)
    pos = lo_ + (hi_ - lo_) * rng.uniform(0.02, 0.98, (32, len(names)))
    for w in range(32):                       # same rule as bn.fit: every walker starts inside the prior
        while not np.isfinite(post(pos[w])):
            pos[w] = lo_ + (hi_ - lo_) * rng.uniform(0.02, 0.98, len(names))
    np.random.seed(7)
    sam = emcee.EnsembleSampler(32, len(names), post)
    sam.run_mcmc(pos, 25000, progress=False)
    ch = sam.get_chain()[::10].astype(np.float32)          # (2500, 32, 12)
    np.savez_compressed(path_tr, chain=ch, names=np.array(names), thin=10, lnp=sam.get_log_prob()[::10].astype(np.float32))
    print("[saved]", path_tr, ch.shape)
tr = np.load(path_tr)
ch = tr["chain"]; half = ch.shape[0] // 2
iA = list(tr["names"]).index("A")
print("A, second half of the chain: median", float(np.median(ch[half:, :, iA])), " sd", float(np.std(ch[half:, :, iA])))


[resume] /Users/gmoretti/Work/wt_bayes/LHCb/Code_general/Bayesian/results/nominal/trace_dispersed.npz
A, second half of the chain: median -0.0440354198217392  sd 0.00856824778020382


## Struttura Dalitz per polarità del magnete

In [4]:
def local_map(lo2, hi2, q, nx=8, ny=8):
    el, eh = np.linspace(0, 16, nx + 1), np.linspace(0, 30, ny + 1)
    npl, _, _ = np.histogram2d(lo2[q > 0], hi2[q > 0], [el, eh]); nmi, _, _ = np.histogram2d(lo2[q < 0], hi2[q < 0], [el, eh])
    n = npl + nmi; ag = (np.sum(q < 0) - np.sum(q > 0)) / len(q)
    with np.errstate(divide="ignore", invalid="ignore"):
        a = (nmi - npl) / n; s = (a - ag) / np.sqrt((1 - a ** 2) / n)
    return n, a, s, ag
lo2 = (np.minimum(data.m12, data.m13).to_numpy() / 1e3) ** 2; hi2 = (np.maximum(data.m12, data.m13).to_numpy() / 1e3) ** 2
mag = data.magnet.to_numpy()
out = {}
maps = {}
for pol in ("all", "down", "up"):
    sel = np.ones(len(q_obs), bool) if pol == "all" else (mag == pol)
    n, a, s, ag = local_map(lo2[sel], hi2[sel], q_obs[sel])
    ok = n >= 20
    from scipy.stats import chi2 as _c2
    c2 = float(np.nansum(s[ok] ** 2)); nd = int(ok.sum() - 1)
    # region of the two |S|>3 bins of the full sample: m2_lo < 4, 7.5 < m2_hi < 11.25 GeV^2
    reg = (lo2[sel] < 4) & (hi2[sel] > 7.5) & (hi2[sel] < 11.25)
    qr = q_obs[sel][reg]; ar = (np.sum(qr < 0) - np.sum(qr > 0)) / len(qr); sr = np.sqrt((1 - ar ** 2) / len(qr))
    out[pol] = {"n": int(sel.sum()), "A_glob": float(ag), "chi2": c2, "ndof": nd, "p": float(_c2.sf(c2, nd)),
                "n_3sigma": int(np.sum(np.abs(s[ok]) > 3)), "region": {"n": int(len(qr)), "A": float(ar), "sigma": float(sr),
                "S_vs_global": float((ar - ag) / sr)}}
    maps[pol] = (n, a, s)
both = (maps["down"][0] >= 20) & (maps["up"][0] >= 20)
r = np.corrcoef(maps["down"][1][both], maps["up"][1][both])[0, 1]
# compatibility of the two polarities bin by bin: (A_down - A_up) / sigma
nd_, ad_ = maps["down"][0][both], maps["down"][1][both]; nu_, au_ = maps["up"][0][both], maps["up"][1][both]
dz = (ad_ - au_) / np.sqrt((1 - ad_ ** 2) / nd_ + (1 - au_ ** 2) / nu_)
c2d = float(np.sum(dz ** 2)); ndd = int(both.sum())
out["down_vs_up"] = {"n_bins": ndd, "pearson_A": float(r), "chi2_difference": c2d, "ndof": ndd, "p": float(_c2.sf(c2d, ndd))}
json.dump(out, open(os.path.join(RESULTS_DIR, "dalitz_magnet.json"), "w"), indent=1)
for k, v in out.items():
    print(k, json.dumps(v))


all {"n": 19731, "A_glob": -0.04323146317976788, "chi2": 80.77695659889034, "ndof": 31, "p": 2.57145541911995e-06, "n_3sigma": 2, "region": {"n": 3091, "A": -0.12779035910708508, "sigma": 0.017839189862752396, "S_vs_global": -4.740063678781357}}
down {"n": 11636, "A_glob": -0.05139223100721898, "chi2": 60.72194559717751, "ndof": 31, "p": 0.0011087680451687342, "n_3sigma": 0, "region": {"n": 1839, "A": -0.12887438825448613, "sigma": 0.02312449922508832, "S_vs_global": -3.3506523316710317}}
up {"n": 8095, "A_glob": -0.03150092649783817, "chi2": 39.52971152958374, "ndof": 28, "p": 0.0727714123519915, "n_3sigma": 1, "region": {"n": 1252, "A": -0.12619808306709265, "sigma": 0.0280357206515586, "S_vs_global": -3.377732206216356}}
down_vs_up {"n_bins": 29, "pearson_A": 0.5044509114158304, "chi2_difference": 22.961535893152366, "ndof": 29, "p": 0.7783005194458201}


## Risonanze: banda della $\phi(1020)$ e regione a bassa massa

In [5]:
m_lo = np.minimum(data.m12, data.m13).to_numpy(); m_hi = np.maximum(data.m12, data.m13).to_numpy()
def cnt(sel):
    qq = q_obs[sel]; a = (np.sum(qq < 0) - np.sum(qq > 0)) / len(qq)
    return {"n": int(len(qq)), "A": float(a), "sigma": float(np.sqrt((1 - a ** 2) / len(qq)))}
PHI = 1019.461   # MeV, world average
phi = np.abs(m_lo - PHI) < 10
reg_sig = ((m_lo / 1e3) ** 2 < 4) & ((m_hi / 1e3) ** 2 > 7.5) & ((m_hi / 1e3) ** 2 < 11.25)   # the two |S|>3 bins
res = {"phi_band": cnt(phi), "phi_sideband_lo2_below_2": cnt(~phi & ((m_lo / 1e3) ** 2 < 2.0)),
       "significant_bins": cnt(reg_sig), "significant_bins_without_phi": cnt(reg_sig & ~phi),
       "significant_bins_phi_only": cnt(reg_sig & phi), "global": cnt(np.ones(len(q_obs), bool))}
h, e = np.histogram(m_lo, np.arange(980, 1100.1, 5))
res["m_lo_hist_980_1100_5MeV"] = h.tolist()
json.dump(res, open(os.path.join(RESULTS_DIR, "dalitz_resonances.json"), "w"), indent=1)
for k, v in res.items():
    print(k, v)


phi_band {'n': 1964, 'A': -0.004073319755600814, 'sigma': 0.02256449692367679}
phi_sideband_lo2_below_2 {'n': 4833, 'A': -0.11980136561142148, 'sigma': 0.01428079694374827}
significant_bins {'n': 3091, 'A': -0.12779035910708508, 'sigma': 0.017839189862752396}
significant_bins_without_phi {'n': 2738, 'A': -0.1446311176040906, 'sigma': 0.018910054292124986}
significant_bins_phi_only {'n': 353, 'A': 0.0028328611898017, 'sigma': 0.05322441597425212}
global {'n': 19731, 'A': -0.04323146317976788, 'sigma': 0.007112450102422257}
m_lo_hist_980_1100_5MeV [0, 9, 44, 39, 60, 84, 172, 905, 690, 201, 120, 97, 76, 68, 64, 56, 82, 59, 87, 57, 60, 59, 58, 60]
